# F1 Pit Stop Prediction — Optimized Notebook

**Goal:** improve training speed and leaderboard performance while keeping the pipeline reproducible.

### Main changes from the original notebook
- Replace the 9-hour AutoGluon `best_quality` search with LightGBM + early stopping.
- Keep `Driver`, `Race`, and `Compound` instead of dropping `Driver`.
- Add race-progress / tyre-age derived features.
- Use a fixed stratified validation split for reproducible model selection.
- Train several LightGBM seeds and average probabilities for a small, fast ensemble.
- Make the external F1 dataset optional instead of always concatenating it.
- Preserve the original submission format exactly.

In [6]:
# Install only if needed (Kaggle usually already has LightGBM)
# !pip install -q lightgbm

import os
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

RANDOM_STATE = 42
LABEL = "PitNextLap"
USE_EXTERNAL_DATA = False  # Turn on only after verifying the external dataset helps validation AUC.
N_ENSEMBLE_MODELS = 1  # Use 3 only if you want a small ensemble after validating the gain.

TRAIN_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\train.csv"
TEST_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\test.csv"
SAMPLE_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\sample_submission.csv"
# EXTERNAL_PATH = "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv"

In [7]:
# Local fallback makes the notebook easy to test outside Kaggle.
if not os.path.exists(TRAIN_PATH):
    TRAIN_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\train.csv"
if not os.path.exists(TEST_PATH):
    TEST_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\test.csv"
if not os.path.exists(SAMPLE_PATH):
    SAMPLE_PATH = r"D:\Work\DSA_Coding_Questions\F1_Pit_prediction\data\sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

print("Train:", train.shape)
print("Test :", test.shape)
print("Target rate:", train[LABEL].mean())

Train: (439140, 16)
Test : (188165, 15)
Target rate: 0.19898210137996994


In [8]:
# Basic cleanup.
# `id` is an identifier, not a predictive feature. Driver/Race are retained.
train = train.drop(columns=["id"], errors="ignore")
test = test.drop(columns=["id"], errors="ignore")
train[LABEL] = train[LABEL].astype(np.int8)

In [ ]:
# # Optional external data.
# # The original notebook always concatenated this dataset. Keep it opt-in because
# # an external source can introduce distribution mismatch or duplicate examples.
# if USE_EXTERNAL_DATA and os.path.exists(EXTERNAL_PATH):
#     external = pd.read_csv(EXTERNAL_PATH)
#     external = external.drop(columns=["Normalized_TyreLife", "id"], errors="ignore")
#     if LABEL not in external.columns:
#         raise ValueError(f"External dataset does not contain {LABEL!r}.")

#     common_cols = [c for c in train.columns if c in external.columns]
#     missing = [c for c in train.columns if c not in external.columns]
#     if missing:
#         raise ValueError(f"External dataset is missing columns: {missing}")

#     external = external[common_cols]
#     external[LABEL] = pd.to_numeric(external[LABEL], errors="raise").astype(np.int8)
#     train = pd.concat([train, external], ignore_index=True)
#     print("External rows added:", len(external))
# else:
#     print("External dataset disabled/not found.")

External dataset disabled/not found.


## Feature engineering

The raw data already contains strong signals. These additional features expose relationships that tree models can learn more easily, especially the amount of race remaining and tyre age relative to the race length.

In [9]:
def add_features(df):
    df = df.copy()

    # Race progress -> approximate total race length and remaining laps.
    progress = df["RaceProgress"].clip(lower=1e-4)
    total_laps = (df["LapNumber"] / progress).replace([np.inf, -np.inf], np.nan)
    df["TotalLaps_est"] = total_laps
    df["LapsRemaining_est"] = (total_laps - df["LapNumber"]).clip(lower=0)

    # Tyre age relative to race length.
    df["TyreLifePct"] = df["TyreLife"] / total_laps.clip(lower=1)

    # Magnitudes often matter more than direction for these signals.
    df["Position_Change_abs"] = df["Position_Change"].abs()
    df["LapTimeDelta_abs"] = df["LapTime_Delta"].abs()

    # Low-cardinality interaction features.
    df["Compound_Stint"] = df["Compound"].astype(str) + "_" + df["Stint"].astype(str)
    df["Year_Compound"] = df["Year"].astype(str) + "_" + df["Compound"].astype(str)
    df["PitStop_Compound"] = df["PitStop"].astype(str) + "_" + df["Compound"].astype(str)

    return df

train = add_features(train)
test = add_features(test)

# Use identical categorical vocabularies for train/test.
cat_cols = train.select_dtypes(include=["object"]).columns.tolist()
for col in cat_cols:
    categories = pd.Index(train[col].astype(str).unique()).union(
        pd.Index(test[col].astype(str).unique())
    )
    train[col] = pd.Categorical(train[col].astype(str), categories=categories)
    test[col] = pd.Categorical(test[col].astype(str), categories=categories)

print("Categorical columns:", cat_cols)
print("Feature count:", train.shape[1] - 1)

Categorical columns: ['Driver', 'Compound', 'Race', 'Compound_Stint', 'Year_Compound', 'PitStop_Compound']
Feature count: 22


In [10]:
X = train.drop(columns=[LABEL])
y = train[LABEL]
X_test = test.copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.15,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_valid.shape)

(373269, 22) (65871, 22)


In [ ]:
# Fast validation model.
# Early stopping prevents wasting time on unnecessary trees.
base_model = LGBMClassifier(
    objective="binary",
    n_estimators=4000,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=40,
    subsample=0.85,
    colsample_bytree=0.90,
    reg_alpha=0.0,
    reg_lambda=2.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=-1,
)

base_model.fit(
    X_train, y_train,
    categorical_feature=cat_cols,
    eval_set=[(X_valid, y_valid)],
    callbacks=[early_stopping(150, verbose=False), log_evaluation(0)],
)

valid_pred = base_model.predict_proba(X_valid)[:, 1]
valid_auc = roc_auc_score(y_valid, valid_pred)
print(f"Validation ROC-AUC: {valid_auc:.6f}")
print(f"Best iteration: {base_model.best_iteration_}")        

Validation ROC-AUC: 0.944248
Best iteration: 768


## Fast ensemble

The notebook defaults to one LightGBM model for maximum speed. If the validation gain is worth the extra runtime, set `N_ENSEMBLE_MODELS = 3` and average the three seeds.

In [12]:
best_iter = int(base_model.best_iteration_ or 800)
seeds = [42, 2026, 3407][:N_ENSEMBLE_MODELS]

valid_preds = []
for seed in seeds:
    model = LGBMClassifier(
        objective="binary",
        n_estimators=best_iter,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_lambda=2.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(X_train, y_train, categorical_feature=cat_cols)
    valid_preds.append(model.predict_proba(X_valid)[:, 1])

ensemble_valid = np.mean(valid_preds, axis=0)
print(f"Single-model ROC-AUC : {roc_auc_score(y_valid, valid_pred):.6f}")
print(f"Ensemble ROC-AUC     : {roc_auc_score(y_valid, ensemble_valid):.6f}")

Single-model ROC-AUC : 0.944248
Ensemble ROC-AUC     : 0.944248


In [13]:
# Refit on all available training rows using the selected number of trees.
final_models = []
for seed in seeds:
    model = LGBMClassifier(
        objective="binary",
        n_estimators=best_iter,
        learning_rate=0.03,
        num_leaves=63,
        min_child_samples=40,
        subsample=0.85,
        colsample_bytree=0.90,
        reg_lambda=2.0,
        random_state=seed,
        n_jobs=-1,
        verbosity=-1,
    )
    model.fit(train.drop(columns=[LABEL]), y, categorical_feature=cat_cols)
    final_models.append(model)

# Average probabilities, which is the correct output for ROC-AUC submissions.
test_pred = np.mean(
    [model.predict_proba(X_test)[:, 1] for model in final_models],
    axis=0,
)

print("Prediction range:", float(test_pred.min()), float(test_pred.max()))

Prediction range: 2.710119356876609e-05 0.9896332951353362


In [14]:
# Submission: preserve the sample_submission id order exactly.
submission = pd.read_csv(SAMPLE_PATH)
submission[LABEL] = test_pred

assert len(submission) == len(test_pred)
assert submission[LABEL].between(0, 1).all()

output_path = "best_quality_optimized.csv"
submission.to_csv(output_path, index=False)
submission.head(), output_path

(       id  PitNextLap
 0  439140    0.001237
 1  439141    0.002289
 2  439142    0.009750
 3  439143    0.102743
 4  439144    0.916329,
 'best_quality_optimized.csv')

## Notes for further tuning

1. If external data improves the same fixed validation split, set `USE_EXTERNAL_DATA=True` and rerun.
2. For a leaderboard push, tune `num_leaves`, `min_child_samples`, `learning_rate`, and `colsample_bytree` around this baseline.
3. Do not optimize the classification threshold: the competition uses ROC-AUC, so probability ranking matters.
4. Keep the validation split fixed while comparing experiments.